### This is to automatically add the cloze pattern to the cloze field

In [15]:
import requests
import json
import re

def invoke(action, **params):
    return requests.post(
        "http://localhost:8765",
        json={
            "action": action,
            "version": 6,
            "params": params
        }
    ).json()

# example: get deck names
print(invoke("deckNames"))


{'result': ['Default', 'French MOVIES', 'Japanese Basic Vokab', 'Japanese Heisig', 'Japanese Heisig::All in one Kanji - Heisig order', 'Japanese Heisig::Deck in progress', 'Japanese Heisig::Remembering the Kanji Vol. 1, 4th Editon', 'Japanese Heisig::Remembering the Kanji Vol. 1, 4th Editon::Part 1: Stories (Lessons 1-12)', 'Japanese Heisig::Remembering the Kanji Vol. 1, 4th Editon::Part 2: Plots (Lessons 13-19)', 'Japanese Heisig::Remembering the Kanji Vol. 1, 4th Editon::Part 3: Elements (Lessons 20-56)', 'Japanese Heisig::Remembering the Kanji Vol. 1, 4th Editon::漢字 Outside 1st Volume', 'Japanese KANJI', 'Japanese KANJI::FirstLove', 'Japanese KANJI::LaLaLand', "Japanese KANJI::YUYU's ポッドキャスト", 'Japanese Media', 'Japanese Media::Filme & Serien', 'Japanese Media::Filme & Serien::FirstLove', 'Japanese Media::Filme & Serien::FirstLove::Episode1 Where the Lilacs bloom', 'Japanese Media::Filme & Serien::FirstLove::Episode2', 'Japanese Media::Filme & Serien::FirstLove::Episode3', 'Japanese

In [24]:
#find all the Notes, that I am already learning in the "Japanese Media" deck, so that I can unsuspend their RTK card.
# take care of the fact that there might be more than one note type in this deck!

note_ids = invoke(
    "findNotes",
    query='deck:"Japanese Media::Youtube::Konnichiwa My Dude Japanese Podcast::ジブリ愛が止まらない | 日本語ポッドキャスト EP299"'
)

#get information about those notes, to find the RTK card ids
notes_info = invoke(
    "notesInfo",
    notes=note_ids['result']
)
#DONT FORGET that in this deck are 2 different note types!!!

#field names
print("Fields in each note:")
for note in notes_info['result'][:1]:  # Print only the first note's fields for inspection
    print(list(note['fields'].keys()))






Fields in each note:
['Cloze', 'Furigana', 'Pitch Accent', 'Word Definition', 'Translation', 'Context Before Cloze', 'Context After Cloze', 'Context', 'Context Translation', 'Notes', 'Grammatik', 'Item Key', 'Subtitle', 'Lemma', 'Word', 'Part of Speech', 'Color', 'Source', 'Language', 'Translation Language', 'Word Transliteration', 'Phrase Transliteration', 'Subtitle Index', 'Item ID', 'Item Title', 'Date Created', 'Date Modified', 'Prev Image', 'Next Image', 'Audio Clip', 'Kanji', 'Diagram', 'Reading', 'Frequency']


### Now we have the note ids and the field names, we can create the cloze deletion and update the notes with the new cloze field.

In [25]:



import re

def make_cloze(cloze_field, lemma, definition):
    # Skip rows where lemma contains a comma
    if "," in lemma:
        return cloze_field  # or return None if you want to filter later

    return re.sub(
        r"<strong>.*?</strong>",
        f"{{{{c1::{lemma}::{definition}}}}}",
        cloze_field,
        count=1
    )



for note in notes_info["result"]:
    note_id = note["noteId"]

    cloze_field = note["fields"]["Cloze"]["value"]  # adjust to your real field name
    Lemma = note["fields"]["Lemma"]["value"]
    Definition = note["fields"]["Notes"]["value"]  # adjust to your real field name

    new_sentence = make_cloze(cloze_field, Lemma, Definition)

    invoke(
        "updateNoteFields",
        note={
            "id": note_id,
            "fields": {
                "Cloze": new_sentence
            }
        }
    )


In [ ]:
# MULTI-LEMMA PROCESSING (added by Codex)
# Splits notes where Lemma contains commas into multiple notes,
# keeps original Cloze context, and creates lemma-specific clozes.

import re


def split_lemmas(lemma_value):
    return [part.strip() for part in lemma_value.split(',') if part.strip()]


def make_cloze_for_lemma(cloze_field, lemma, definition):
    # Prefer replacing the <strong> tag that contains this exact lemma.
    specific_pattern = re.compile(rf"<strong>\s*{re.escape(lemma)}\s*</strong>")
    replacement = f"{{{{c1::{lemma}::{definition}}}}}"

    if specific_pattern.search(cloze_field):
        return specific_pattern.sub(replacement, cloze_field, count=1)

    # Fallback if exact strong match is not found.
    return re.sub(r"<strong>.*?</strong>", replacement, cloze_field, count=1)


def get_note_fields(note):
    return {name: meta['value'] for name, meta in note['fields'].items()}


def get_deck_name_for_note(note):
    if note.get("deckName"):
        return note["deckName"]

    card_ids = note.get("cards", [])
    if not card_ids:
        return None

    cards_info = invoke("cardsInfo", cards=[card_ids[0]])
    if cards_info.get("error") or not cards_info.get("result"):
        return None

    return cards_info["result"][0].get("deckName")


for note in notes_info["result"]:
    note_id = note["noteId"]
    original_cloze = note["fields"]["Cloze"]["value"]
    lemma_value = note["fields"]["Lemma"]["value"]
    definition = note["fields"]["Notes"]["value"]

    lemmas = split_lemmas(lemma_value)

    if len(lemmas) <= 1:
        lemma = lemmas[0] if lemmas else lemma_value.strip()
        new_cloze = make_cloze_for_lemma(original_cloze, lemma, definition)
        invoke(
            "updateNoteFields",
            note={
                "id": note_id,
                "fields": {
                    "Lemma": lemma,
                    "Subtitle": original_cloze,
                    "Cloze": new_cloze,
                },
            },
        )
        continue

    # Keep first lemma on original note.
    first_lemma = lemmas[0]
    first_cloze = make_cloze_for_lemma(original_cloze, first_lemma, definition)
    invoke(
        "updateNoteFields",
        note={
            "id": note_id,
            "fields": {
                "Lemma": first_lemma,
                "Subtitle": original_cloze,
                "Cloze": first_cloze,
            },
        },
    )

    # Clone one new note per extra lemma.
    base_fields = get_note_fields(note)
    deck_name = get_deck_name_for_note(note)
    if not deck_name:
        print(f"Could not determine deck for note {note_id}; skipping clones")
        continue

    for extra_lemma in lemmas[1:]:
        cloned_fields = dict(base_fields)
        cloned_fields["Lemma"] = extra_lemma
        cloned_fields["Subtitle"] = original_cloze
        cloned_fields["Cloze"] = make_cloze_for_lemma(original_cloze, extra_lemma, definition)

        result = invoke(
            "addNote",
            note={
                "deckName": deck_name,
                "modelName": note["modelName"],
                "fields": cloned_fields,
                "tags": note.get("tags", []),
            },
        )
        if result.get("error"):
            print(f"Failed to clone note {note_id} for lemma '{extra_lemma}': {result['error']}")
        else:
            print(f"Created note {result['result']} from note {note_id} for lemma '{extra_lemma}'")




In [ ]:
# AI DEFINITION UPDATE FOR TAGGED NOTES
# Updates definition text for notes tagged: two_words_in_furigana_field
# Provider options:
# - provider = "openai"  (requires OPENAI_API_KEY)
# - provider = "ollama"  (free/local, requires Ollama running)

import os
import json
import requests

provider = "ollama"            # "openai" or "ollama"
openai_model = "gpt-4o-mini"
ollama_model = "qwen2.5:7b"    # change if needed, e.g. "llama3.1:8b"
dry_run = True                  # set False to write updates to Anki
limit = 50                      # max notes to process this run

TARGET_TAG = "two_words_in_furigana_field"


def ai_rewrite_definition(old_definition, lemma, furigana, cloze, notes_field, word_definition):
    prompt = f"""You are improving Japanese vocabulary flashcard definitions.

Return only one concise definition sentence/line.
Keep the meaning accurate and learner-friendly.
If old definition is already good, lightly clean it.
Do not include markdown, bullets, or explanations.

Lemma: {lemma}
Furigana: {furigana}
Cloze sentence: {cloze}
Old definition: {old_definition}
Notes field: {notes_field}
Word Definition field: {word_definition}
"""

    if provider == "openai":
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise RuntimeError("OPENAI_API_KEY is not set")

        r = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
            json={
                "model": openai_model,
                "messages": [
                    {"role": "system", "content": "Rewrite definitions clearly and briefly."},
                    {"role": "user", "content": prompt},
                ],
                "temperature": 0.2,
            },
            timeout=60,
        )
        r.raise_for_status()
        data = r.json()
        return data["choices"][0]["message"]["content"].strip()

    if provider == "ollama":
        r = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": ollama_model,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.2},
            },
            timeout=120,
        )
        r.raise_for_status()
        data = r.json()
        return data.get("response", "").strip()

    raise ValueError(f"Unknown provider: {provider}")


note_ids = invoke("findNotes", query=f'tag:{TARGET_TAG}')
if note_ids.get("error"):
    raise RuntimeError(note_ids["error"])

ids = note_ids.get("result", [])[:limit]
print(f"Found {len(note_ids.get('result', []))} notes with tag '{TARGET_TAG}'. Processing {len(ids)}.")

if not ids:
    print("No notes to process.")
else:
    notes_info = invoke("notesInfo", notes=ids)
    if notes_info.get("error"):
        raise RuntimeError(notes_info["error"])

    updated = 0
    skipped = 0

    for note in notes_info.get("result", []):
        note_id = note["noteId"]
        fields = note.get("fields", {})

        lemma = fields.get("Lemma", {}).get("value", "").strip()
        furigana = fields.get("Furigana", {}).get("value", "").strip()
        cloze = fields.get("Cloze", {}).get("value", "").strip()
        notes_field = fields.get("Notes", {}).get("value", "").strip()
        word_definition = fields.get("Word Definition", {}).get("value", "").strip()

        # Prefer Notes as target; fallback to Word Definition if Notes missing.
        if "Notes" in fields:
            target_field = "Notes"
            old_definition = notes_field
        elif "Word Definition" in fields:
            target_field = "Word Definition"
            old_definition = word_definition
        else:
            print(f"Skip {note_id}: no Notes/Word Definition field")
            skipped += 1
            continue

        try:
            new_definition = ai_rewrite_definition(
                old_definition=old_definition,
                lemma=lemma,
                furigana=furigana,
                cloze=cloze,
                notes_field=notes_field,
                word_definition=word_definition,
            )
        except Exception as e:
            print(f"Skip {note_id}: AI call failed -> {e}")
            skipped += 1
            continue

        if not new_definition or new_definition == old_definition:
            print(f"No change {note_id}")
            skipped += 1
            continue

        print(f"Update {note_id} ({target_field})")
        print(f"  OLD: {old_definition[:120]}")
        print(f"  NEW: {new_definition[:120]}")

        if not dry_run:
            res = invoke(
                "updateNoteFields",
                note={
                    "id": note_id,
                    "fields": {target_field: new_definition},
                },
            )
            if res.get("error"):
                print(f"  Failed update {note_id}: {res['error']}")
                skipped += 1
                continue

        updated += 1

    print(json.dumps({"updated": updated, "skipped": skipped, "dry_run": dry_run}, ensure_ascii=False))



# query all the notes in the first love deck, which have an empty lemma field. 
# put everything inside 'Word Definition' into 'Notes'# 
# extract 'Lemma' and 'Word Definition' from the cloze (cloze looks like this “人生は{{c1::まるで::as if, quite}} ジグソーパズルだ”と)


In [2]:
#!/usr/bin/env python3
import json
import re
import urllib.request

ANKI_CONNECT_URL = "http://localhost:8765"
DECK_NAME = "Japanese Media::Filme & Serien::FirstLove"  # change if needed
DRY_RUN = True  # set to False to write changes

# Matches: {{c1::まるで::as if, quite}}
CLOZE_RE = re.compile(r"\{\{c\d+::(.*?)(?:::([^}]*))?\}\}", re.DOTALL)


def invoke(action, **params):
    payload = {"action": action, "version": 6, "params": params}
    req = urllib.request.Request(
        ANKI_CONNECT_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    if data.get("error"):
        raise RuntimeError(f"AnkiConnect error ({action}): {data['error']}")
    return data["result"]


def merge_notes_with_word_definition(notes_val: str, word_def_val: str) -> str:
    notes_val = (notes_val or "").strip()
    word_def_val = (word_def_val or "").strip()

    if not word_def_val:
        return notes_val
    if not notes_val:
        return word_def_val
    if word_def_val in notes_val:
        return notes_val
    return f"{notes_val}<br>{word_def_val}"


def extract_from_cloze(cloze_text: str):
    if not cloze_text:
        return None, None
    m = CLOZE_RE.search(cloze_text)
    if not m:
        return None, None
    lemma = (m.group(1) or "").strip()
    definition = (m.group(2) or "").strip()
    return lemma, definition


def main():
    query = f'deck:"{DECK_NAME}" "Lemma:"'
    note_ids = invoke("findNotes", query=query)
    print(f"Found {len(note_ids)} notes with empty Lemma in deck: {DECK_NAME}")

    if not note_ids:
        return

    notes_info = invoke("notesInfo", notes=note_ids)

    updated = 0
    skipped = 0

    for note in notes_info:
        nid = note["noteId"]
        fields = note.get("fields", {})

        cloze_val = fields.get("Cloze", {}).get("value", "")
        old_word_def = fields.get("Word Definition", {}).get("value", "")
        old_notes = fields.get("Notes", {}).get("value", "")

        new_notes = merge_notes_with_word_definition(old_notes, old_word_def)
        new_lemma, new_word_def = extract_from_cloze(cloze_val)

        if not new_lemma and not new_word_def and new_notes == old_notes:
            skipped += 1
            print(f"Skip {nid}: no cloze extraction and no Notes update")
            continue

        update_fields = {}
        if new_notes != old_notes:
            update_fields["Notes"] = new_notes
        if new_lemma:
            update_fields["Lemma"] = new_lemma
        if new_word_def is not None:
            update_fields["Word Definition"] = new_word_def  # may become empty string

        if not update_fields:
            skipped += 1
            print(f"Skip {nid}: nothing to update")
            continue

        print(f"Update {nid}: {list(update_fields.keys())}")
        if not DRY_RUN:
            invoke("updateNoteFields", note={"id": nid, "fields": update_fields})
        updated += 1

    print(f"Done. Updated: {updated}, Skipped: {skipped}, Dry run: {DRY_RUN}")


if __name__ == "__main__":
    main()


Found 454 notes with empty Lemma in deck: Japanese Media::Filme & Serien::FirstLove
Update 1734510465438: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465439: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465448: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465451: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465453: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465454: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465455: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465456: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465457: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465459: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465460: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465466: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465467: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465468: ['Notes', 'Lemma', 'Word Definition']
Update 1734510465473: ['Notes', 'Lemma', 'Word Definition']
Update 173451046